In [1]:
import matplotlib.pyplot as plt
import numpy as np
import PIL
import tensorflow as tf
import opendatasets as od 
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
import pandas as pd
import os
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn import tree
import random
from PIL import Image
import cv2
import opendatasets as od 

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

In [25]:
od.download(
    "https://www.kaggle.com/datasets/andrewmvd/ocular-disease-recognition-odir5k")

df = pd.read_excel("Processed_data.xlsx")
df.head()
#path_to_images = "./ocular-disease-recognition-odir5k/"

Skipping, found downloaded files in ".\ocular-disease-recognition-odir5k" (use force=True to force download)


,Patient ID,Patient Age,Patient Sex,Filename,Diagnosis,N,D,G,C,A,H,M,O
0,0,69,Female,0_left.jpg,cataract,0,0,0,1,0,0,0,0
1,1,57,Male,1_left.jpg,normal fundus,1,0,0,0,0,0,0,0
2,2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy",0,1,0,0,0,0,0,1
3,3,66,Male,3_left.jpg,normal fundus,1,0,0,0,0,0,0,0
4,4,53,Male,4_left.jpg,macular epiretinal membrane,0,0,0,0,0,0,0,1


In [26]:
# Creating the target Labels 
def createTarget(df):
    df['Labels'] = [[0,0,0,0,0,0,0] for i in range (0,len(df))] #create a column for labels with 8 0s
    target_columns = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
    label = [] #create empty label array
    for i in range(0, len(df)):
        for target in target_columns:
            label.append(df.loc[i, target]) #append the value for the column for each row to the label
        
        df.at[i,'Labels'] = label #update the label column with the label array
        label = [] #reset label array
            

In [27]:
createTarget(df)

In [28]:
res_df = df.drop(['Patient ID', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O'], axis=1)
res_df

,Patient Age,Patient Sex,Filename,Diagnosis,Labels
0,69,Female,0_left.jpg,cataract,"[0, 0, 0, 1, 0, 0, 0, 0]"
1,57,Male,1_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
2,42,Male,2_left.jpg,"laser spot,moderate non proliferative retinopathy","[0, 1, 0, 0, 0, 0, 0, 1]"
3,66,Male,3_left.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
4,53,Male,4_left.jpg,macular epiretinal membrane,"[0, 0, 0, 0, 0, 0, 0, 1]"
...,...,...,...,...,...
6995,63,Male,4686_right.jpg,proliferative diabetic retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6996,42,Male,4688_right.jpg,moderate non proliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"
6997,54,Male,4689_right.jpg,normal fundus,"[1, 0, 0, 0, 0, 0, 0, 0]"
6998,57,Male,4690_right.jpg,mild nonproliferative retinopathy,"[0, 1, 0, 0, 0, 0, 0, 0]"


In [29]:
def binary_to_decimal(binary_list):
    """
    Converts a list of 0s and 1s (binary) to its decimal equivalent.

    :param binary_list: List of 0s and 1s.
    :return: Decimal equivalent of the binary number.
    """
    return sum(val * (2 ** idx) for idx, val in enumerate(reversed(binary_list)))


In [30]:
# Apply the function to each row
res_df['target'] = res_df['Labels'].apply(binary_to_decimal)

print(res_df)

      Patient Age Patient Sex        Filename  \
0              69      Female      0_left.jpg   
1              57        Male      1_left.jpg   
2              42        Male      2_left.jpg   
3              66        Male      3_left.jpg   
4              53        Male      4_left.jpg   
...           ...         ...             ...   
6995           63        Male  4686_right.jpg   
6996           42        Male  4688_right.jpg   
6997           54        Male  4689_right.jpg   
6998           57        Male  4690_right.jpg   
6999           58        Male  4784_right.jpg   

                                              Diagnosis  \
0                                              cataract   
1                                         normal fundus   
2     laser spot,moderate non proliferative retinopathy   
3                                         normal fundus   
4                           macular epiretinal membrane   
...                                                 ...  

In [31]:
print(res_df)

      Patient Age Patient Sex        Filename  \
0              69      Female      0_left.jpg   
1              57        Male      1_left.jpg   
2              42        Male      2_left.jpg   
3              66        Male      3_left.jpg   
4              53        Male      4_left.jpg   
...           ...         ...             ...   
6995           63        Male  4686_right.jpg   
6996           42        Male  4688_right.jpg   
6997           54        Male  4689_right.jpg   
6998           57        Male  4690_right.jpg   
6999           58        Male  4784_right.jpg   

                                              Diagnosis  \
0                                              cataract   
1                                         normal fundus   
2     laser spot,moderate non proliferative retinopathy   
3                                         normal fundus   
4                           macular epiretinal membrane   
...                                                 ...  

In [32]:
# Path to your image dataset
image_dataset_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Training images"
validation_path = "./ocular-disease-recognition-odir5k/ODIR-5K/ODIR-5K/Testing images"

In [33]:
image_paths = res_df['Filename'].apply(lambda x: os.path.join(image_dataset_path, x))  # Replace with the correct image folder path
labels = res_df['target']
image_paths
labels

0        16
1       128
2        65
3       128
4         1
       ... 
6995     64
6996     64
6997    128
6998     64
6999     12
Name: target, Length: 7000, dtype: int64

In [34]:
# # Initialize the lists to hold the feature vectors and labels
# features = []
# labels = []

# # Process the images and labels:
# for index, row in df.iterrows():
#     # Open the image
#     img_path = os.path.join(image_dataset_path, row['Filename'])
#     with Image.open(img_path) as img:
        
#         img = img.resize((128, 128))  #resizing 
#         img_array = np.array(img)  # Convert to numpy array
#         img_array = img_array.flatten()  # Flatten the image array
    
#     # Append the age to the image array.
#     patient_age = float(row['Patient Age'])  # Cast age to float
#     feature_with_age = np.append(img_array, patient_age)  # Append age to the feature array

#     # Append the feature_with_age array to the features list
#     features.append(feature_with_age)
    
#     # Append the label (which is already a list) to the labels list
#     labels.append(row['Labels'])

# # Convert lists to numpy arrays
# features = np.array(features)
# labels = np.array(labels)

In [35]:
label_encoder = LabelEncoder()
numeric_labels = label_encoder.fit_transform(labels)

In [36]:
def load_and_preprocess_image(path, img_height, img_width):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [img_height, img_width])
    img = img / 255.0  # Normalize to [0,1]
    return img

In [37]:
path_ds = tf.data.Dataset.from_tensor_slices(image_paths)
label_ds = tf.data.Dataset.from_tensor_slices(numeric_labels)


In [38]:
# Constants
img_height = 128  # Replace with your value
img_width = 128   # Replace with your value
batch_size = 8   # Replace with your value

# Function to load and process images in batches
def load_and_preprocess_from_path_label(path, label):
    return load_and_preprocess_image(path, img_height, img_width), label

# Map the function to the dataset
dataset = tf.data.Dataset.zip((path_ds, label_ds))
dataset = dataset.map(load_and_preprocess_from_path_label)


In [39]:
dataset = dataset.shuffle(buffer_size=len(image_paths))
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)


In [40]:
# num_classes = len(class_names)

model = Sequential([
  layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)),
  layers.Conv2D(16, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(32, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Conv2D(64, 3, padding='same', activation='relu'),
  layers.MaxPooling2D(),
  layers.Flatten(),
  layers.Dense(128, activation='relu'),
  Dense(len(label_encoder.classes_), activation='softmax') 
])





# model = Sequential([
#     Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
#     MaxPooling2D(2, 2),
#     Conv2D(64, (3, 3), activation='relu'),
#     MaxPooling2D(2, 2),
#     Conv2D(128, (3, 3), activation='relu'),
#     MaxPooling2D(2, 2),
#     Flatten(),
#     Dense(128, activation='relu'),
#     Dropout(0.5),
#     Dense(len(label_encoder.classes_), activation='softmax')  # Number of classes
# ])







In [44]:
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


In [45]:
history = model.fit(
    dataset,  # Your training dataset
    epochs=10,  # Number of epochs
    # validation_data=validation_dataset,  # Your validation dataset
    verbose=1
)


Epoch 1/10
875/875 [==============================] - 310s 253ms/step - loss: 2.1251 - accuracy: 0.4454
Epoch 2/10
875/875 [==============================] - 361s 260ms/step - loss: 1.8976 - accuracy: 0.4454
Epoch 3/10
875/875 [==============================] - 337s 288ms/step - loss: 1.8850 - accuracy: 0.4454
Epoch 4/10
875/875 [==============================] - 351s 263ms/step - loss: 1.8818 - accuracy: 0.4454
Epoch 5/10
875/875 [==============================] - 309s 259ms/step - loss: 1.8801 - accuracy: 0.4454
Epoch 6/10
875/875 [==============================] - 350s 261ms/step - loss: 1.8793 - accuracy: 0.4454
Epoch 7/10
875/875 [==============================] - 332s 279ms/step - loss: 1.8791 - accuracy: 0.4454
Epoch 8/10
875/875 [==============================] - 318s 270ms/step - loss: 1.8788 - accuracy: 0.4454
Epoch 9/10
875/875 [==============================] - 387s 302ms/step - loss: 1.8779 - accuracy: 0.4454
Epoch 10/10
875/875 [==============================] - 352s 292m

In [ ]:
def binary_to_decimal(binary_list):
    """
    Converts a list of 0s and 1s (binary) to its decimal equivalent.

    :param binary_list: List of 0s and 1s.
    :return: Decimal equivalent of the binary number.
    """
    return sum(val * (2 ** idx) for idx, val in enumerate(reversed(binary_list)))


In [ ]:
import pandas as pd

# Example DataFrame
data = {'binary': [[1, 0, 1], [1, 1, 0], [1, 0, 0, 1]]}
df = pd.DataFrame(data)

# Apply the function to each row
df['decimal'] = df['binary'].apply(binary_to_decimal)

print(df)


In [ ]:
# one hot encoder object to be used for binned age group
one_hot = OneHotEncoder()

# label encoder to be used for sex
label_encoder = LabelEncoder() 

# multi label binarizer object to be used for diagnosis
mlb = MultiLabelBinarizer() 

In [5]:
path_to_images = "./ocular-disease-recognition-odir5k/"
training_path = "ODIR-5k/ODIR-5k/Training Images"
test_path = "ODIR-5k/ODIR-5k/Testing Images"
preprocessed_images = "preprocessed_images"

print("Training Images Length: ", len(os.listdir(os.path.join(path_to_images, training_path))))
print("Testing Images Length: ", len(os.listdir(os.path.join(path_to_images, test_path))))
print("Preprocessed Images Length: ", len(os.listdir(os.path.join(path_to_images, preprocessed_images))))

Training Images Length:  7000
Testing Images Length:  1000
Preprocessed Images Length:  6392


In [6]:
batch_size = 32
img_height = 180
img_width = 180


In [28]:
path_to_training= os.listdir(os.path.join(path_to_images, training_path))
path_name = "ocular-disease-recognition-odir5k/ODIR-5k/ODIR-5k/Training Images"
#path_to_training

In [37]:
if os.path.isdir(path_name):
    print("Directory exists.")
    if any(file.endswith(('.bmp', '.gif', '.jpeg', '.jpg', '.png')) for file in os.listdir(path_name)):
        print("Images found in the directory.")
    else:
        print("No images found in the allowed formats.")
else:
    print("Directory does not exist.")


Directory exists.
Images found in the directory.


Found 0 files belonging to 0 classes.
Using 0 files for training.


ValueError: No images found in directory ocular-disease-recognition-odir5k/ODIR-5k/ODIR-5k/Training Images. Allowed formats: ('.bmp', '.gif', '.jpeg', '.jpg', '.png')